# Cora edge-recovery stability — Colab runner

Runs `configs/cora.toml` (leakage-free split, cumulative damage series, p_e distributions) on a GPU
and saves every run to Google Drive. All parameters live in the config file; the resolved config,
git commit and library versions are saved inside each run directory.

**Runtime → Change runtime type → GPU** before running.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = "https://github.com/IritKogan/HypDiff_Network-Stability.git"
BRANCH = "phase-b-work"
REPO_DIR = "/content/HypDiff_Network-Stability"
RUNS_DIR = "/content/drive/MyDrive/hypdiff_runs"      # results + checkpoints
DATA_DIR = "/content/drive/MyDrive/hypdiff_data"      # cached datasets
CONFIG = "configs/cora.toml"

In [ ]:
import os, subprocess
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
%cd {REPO_DIR}
!git log --oneline -1

Install the pinned versions (`requirements-phaseb.txt`). Different networkx/numpy versions change generated graphs and seeds.

In [ ]:
!pip install -q -r requirements-phaseb.txt pytest
import torch, torch_geometric, networkx, numpy
print(torch.__version__, torch_geometric.__version__, networkx.__version__, numpy.__version__)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

Keep datasets on Drive so they are downloaded only once.

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)
if not os.path.islink("data/Cora") and not os.path.exists("data/Cora"):
    os.makedirs(f"{DATA_DIR}/Cora", exist_ok=True)
    os.symlink(f"{DATA_DIR}/Cora", "data/Cora")

Run the test suite first (CPU parts, ~10 s).

In [ ]:
!python -m pytest -q -p no:warnings

Run the experiment. `--output-root` and `--device` are recorded in the run's `config.json`.

In [ ]:
!python scripts/run_experiment.py --config {CONFIG} --output-root {RUNS_DIR} --device cuda --log-every 250

In [ ]:
import glob, json
from IPython.display import Image, display
run = sorted(glob.glob(f"{RUNS_DIR}/cora_edge_recovery_*"))[-1]
print(run)
summary = json.load(open(f"{run}/summary.json"))
print("best val AUC:", summary["training"]["autoencoder_best_val_auc"],
      "| NMI clusters vs classes:", summary["training"]["community_nmi_vs_classes"])
for attack, levels in summary["attacks"].items():
    for e in levels:
        print(f"{attack:24s} {e['damage_fraction']:.0%}  p_e mean {e['p_e']['mean']:.3f}  "
              f"no-diffusion {e['p_e_no_diffusion']['mean']:.3f}  control FP {e['control_false_positive']['mean']:.4f}")
display(Image(f"{run}/figures/training_curves.png"))
display(Image(f"{run}/figures/p_e_distributions.png"))